# Reinforcement Learning

# 2. Dynamic programming

This notebook presents policy iteration and value iteration for finding the optimal policy.

Note that these techniques require the enumeration of all states and thus apply to a few models only (e.g., walk, maze, Tic-Tac-Toe, Nim).

In [ ]:
import numpy as np

In [ ]:
from model import Walk, Maze, TicTacToe, Nim
from agent import Agent

## Walk

In [ ]:
walk = Walk()

In [ ]:
states = walk.get_all_states()

In [ ]:
len(states)

## Maze

In [ ]:
maze_map = np.load('maze.npy')

In [ ]:
maze = Maze()
init_state = (1, 0)
exit_state = (1, 20)
maze.set_parameters(maze_map, init_state, [exit_state])
maze = Maze()

In [ ]:
states = maze.get_all_states()

In [ ]:
len(states)

In [ ]:
maze.display()

## Policy Iteration

In policy iteration, you start from an arbitrary policy and improve it sequentially from its value function. The limiting policy is optimal.

In [ ]:
from dynamic import PolicyEvaluation, PolicyIteration

In [ ]:
# let's start with the random policy
agent = Agent(maze)
policy = agent.policy

In [ ]:
# policy evaluation
algo = PolicyEvaluation(maze, policy)
algo.evaluate_policy()
values = algo.values

In [ ]:
len(values)

In [ ]:
maze.display_values(values)

In [ ]:
# policy improvement
new_policy = algo.get_policy()

In [ ]:
maze.display_policy(new_policy)

In [ ]:
# let's test this new policy
agent = Agent(maze, new_policy)
stop, states, rewards = agent.get_episode()

In [ ]:
animation = maze.display(states)

In [ ]:
animation

In general, several iterations of policy evaluation / policy improvement are necessary. 

In [ ]:
algo = PolicyIteration(maze)

In [ ]:
policy = algo.get_optimal_policy()
maze.display_policy(policy)

In [ ]:
values = algo.values
maze.display_values(values)

## To do

Consider the Walk environment with a discount factor $\gamma = 0.9$.
* What is the expected gain of a random walk?
* Compare with the expected gain of the optimal policy, obtained by Policy Iteration.
* Display the optimal value function and the optimal policy. Interpret the results.
* Increase the strength of the wind and observe the new results.

In [ ]:
gamma = 0.9

**Random walk**

In [ ]:
walk = Walk()
agent = Agent(walk)

policy = agent.policy
algo = PolicyEvaluation(walk, policy, gamma=gamma)
algo.evaluate_policy()

rw_gains = agent.get_gains(gamma=gamma)
print("Gains of a random walk: {}".format(np.mean(rw_gains)))

In [ ]:
values = algo.values
walk.display_policy(policy)
walk.display_values(values)

**Expected gain of optimal policy, obtained by Policy Iteration**

In [ ]:
algo = PolicyIteration(walk, gamma=gamma)
policy = algo.get_optimal_policy()

agent = Agent(walk, policy)
op_gains = agent.get_gains(gamma=gamma)
print("Gains of an optimal walk: {}".format(np.mean(op_gains)))
print("Difference in gains: {}".format(np.mean(op_gains) - np.mean(rw_gains)))

The optimal policy outperforms the random walk in gains. This is because the optimal policy prioritizes states that maximize rewards while accounting for the discount factor $(\gamma = 0.9)$. The random walk explores the environment randomly, leading to not optimal outcomes.

**Display of the optimal value function and the optimal policy**

In [ ]:
values = algo.values
walk.display_policy(policy)
walk.display_values(values)

In [ ]:
# Simulation of the optimal policy from each state and get the average gain

gains_per_state = []
for state in walk.get_all_states():
    walk.state = state
    agent = Agent(walk, policy)
    gains_per_state.append(np.mean(agent.get_gains(state=state, n_runs=20, gamma=gamma)))

walk.display_values(gains_per_state)

**Interpretation**

With a discount factor of $\gamma = 0.9$ the agent considers future rewards. This explains why states farther from (3, 3) have lower values but still shows a tendency to move towards the high-reward state. The optimal value function shows that the 4 states next to (3, 3) have the highest expected gains, this is because starting from these the policy indicates the agent to move to the reward.

The agent is willing to pass through states with negative rewards if it leads to a higher cumulative reward in the future ((1,3) and (3,1)). This shows that the value function takes into account both immediate and future rewards.

Additionally, by simulating 20 independent runs starting from each state and averaging the gains, we observe a pattern that mirrors the optimal value function. This indicates that the value function correctly estimates the expected gains and shows how the agent performs starting from different states in the walk.

**Increased wind**

In [ ]:
# previous wind 
wind = walk.Wind

# new wind
wind_ = {(0, 1): 0.8, (1, 0): 0.1}

Walk.set_parameters(Walk.Size, Walk.Rewards, wind_)

In [ ]:
walk_ = Walk()
algo = PolicyIteration(walk_, gamma=gamma)
policy = algo.get_optimal_policy()

agent = Agent(walk, policy)
wind_gains = agent.get_gains(gamma=gamma)
print("Gains of an optimal walk with wind: {}".format(np.mean(wind_gains)))
print("Difference in gains: {}".format(np.mean(wind_gains) - np.mean(op_gains)))

In [ ]:
values = algo.values
walk.display_policy(policy)
walk.display_values(values)

**Interpretation**

The new wind introduces a high probability of moving the agent to the right, after the action taken by the agent. As a result, the policy now prioritizes moving towards state (3, 2), since the wind will most likely push the agent further to the right when it attempts to reach that state. This behavior can be observed in the optimal value function, where states around (3, 2) show the highest expected gains. Additionally, other high-value states are those that are one step away from these high-value regions, indicating a tendency to move towards high-reward states.

As observed in the previous setup, the model is still willing to pass through states with negative rewards if it ultimately leads to reaching the highest reward at (3, 3). We can observe that even in an environment which is not deterministic, policy iteration is still useful for optimizing gains.

## Value Iteration

Value iteration relies on Bellman's optimality equation. 

## To do

Check the code of ``ValueIteration`` below.
* Complete the method ``get_optimal_policy``.
* Test it on the maze and the walk.
* You play TicTacToe at random against an adversary using the one-step policy. What is your expected gain? 
* Observe the improvement when you play perfectly against the same adversary.
* Do the same with Nim.

In [ ]:
class ValueIteration(PolicyEvaluation):
    """Value iteration. 
    
    Parameters
    ----------
    model: object of class Environment
        The model.
    player: int
        Player for games (1 or -1, default = default player of the game).        
    gamma: float
        Discount factor (between 0 and 1).
    n_iter: int
        Maximum number of value iterations.
    """
    
    def __init__(self, model, player=None, gamma=1, n_iter=100):
        agent = Agent(model, player=player)
        policy = agent.policy
        player = agent.player
        super(ValueIteration, self).__init__(model, policy, player, gamma)  
        self.n_iter = n_iter
        
    def get_optimal_policy(self):
        """Get the optimal policy by iteration of Bellman's optimality equation."""
        transitions = self.transitions
        # Bellman's optimality equation
        values = np.zeros(self.n_states)
        for t in range(self.n_iter):    
            next_values = self.rewards + self.gamma * values
            action_value = {action: transition.dot(next_values) for action, transition in self.transitions.items()}
            values = np.zeros(self.n_states)
            for i, state in enumerate(self.states):
                if not self.model.is_terminal(state):
                    actions = self.get_actions(state)
                    # update values
                    values[i] = max(action_value[action][i] for action in actions)
        self.values = values
        policy = self.get_policy()
        return policy


**Test value iteration on Walk**

In [ ]:
algo = ValueIteration(walk)

In [ ]:
policy = algo.get_optimal_policy()
walk.display_policy(policy)

values = algo.values
walk.display_values(values)

**Test value iteration on Maze**

In [ ]:
algo = ValueIteration(maze)

In [ ]:
policy = algo.get_optimal_policy()
maze.display_policy(policy)

values = algo.values
maze.display_values(values)

**TicTacToe at random vs one-step policy adversary**

In [ ]:
game = TicTacToe(adversary_policy='one_step')
agent = Agent(game, policy='random')

In [ ]:
gains = agent.get_gains()
print("Gains of a random player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The expected gain of playing TicTacToe randomly against a better adversary (one-step policy) is negative. Since we will lose most of the times, tie some of the times and win very few times. This is because we have a significant decision making process because of the selected policies.

**TicTacToe perfect vs one-step policy adversary**

In [ ]:
game = TicTacToe(adversary_policy='one_step')

algo = ValueIteration(game)
vi_optimal_policy = algo.get_optimal_policy()

agent = Agent(game, policy=vi_optimal_policy)

In [ ]:
gains = agent.get_gains()
print("Gains of a perfect player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The expected gain of playing perfect TicTacToe against a one-step policy adversary is positive (almost all wins). The perfect player never losses and only draws ~10% of games played.

**Nim at random vs one-step policy adversary**

In [ ]:
game = Nim(adversary_policy='one_step')
agent = Agent(game, policy='random')

In [ ]:
gains = agent.get_gains()
print("Gains of a random player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The expected gain of playing TicTacToe randomly against a better adversary (one-step policy) is almost 0. On comparison to TicTacToe, the one-step policy does not seem to give a big advantage to the adversary in comparison with the random policy

**Nim perfect vs one-step policy adversary**

In [ ]:
game = Nim(adversary_policy='one_step')

algo = ValueIteration(game)
vi_optimal_policy = algo.get_optimal_policy()

agent = Agent(game, policy=vi_optimal_policy)

In [ ]:
gains = agent.get_gains()
print("Gains of a perfect player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The expected gain of playing perfect Nim against a one-step policy adversary is positive. The perfect player always wins against the one-step policy adversary.

## Perfect players

We now use Value Iteration to get perfect players, assuming the best response of the adversary.

## To do

Check the code of the new class ``ValueIteration`` below.
* Complete the method ``get_perfect_players``.
* Test it on TicTacToe. Who wins?
* Test it on Nim. Who wins?
* Is this approach applicable to ConnectFour? Why?

In [ ]:
from scipy import sparse

In [ ]:
class ValueIteration(PolicyEvaluation):
    """Value iteration. 
    
    Parameters
    ----------
    model: object of class Environment
        The model.
    player: int
        Player for games (1 or -1, default = default player of the game).        
    gamma: float
        Discount factor (between 0 and 1).
    n_iter: int
        Maximum number of value iterations.
    """
    
    def __init__(self, model, player=None, gamma=1, n_iter=100):
        agent = Agent(model, player=player)
        policy = agent.policy
        player = agent.player
        super(ValueIteration, self).__init__(model, policy, player, gamma)  
        self.n_iter = n_iter
    
    def get_perfect_players(self):
        """Get perfect players for games, with the best response of the adversary."""
        if not self.model.is_game():
            raise ValueError("This method applies to games only.")
        # get transitions for each player
        actions = self.model.get_all_actions()
        transitions = {action: sparse.lil_matrix((self.n_states, self.n_states)) for action in actions}
        for i, state in enumerate(self.states):    
            actions = self.model.get_available_actions(state)
            for action in actions:
                next_state = self.model.get_next_state(state, action)
                j = self.get_state_id(next_state)
                transitions[action][i, j] = 1
        transitions = {action: sparse.csr_matrix(transition) for action, transition in transitions.items()}
        self.transitions = transitions
        # Bellman's optimality equation
        values = np.zeros(self.n_states)
        # to be completed
        for t in range(self.n_iter):    
            next_values = self.rewards + self.gamma * values
            action_value = {action: transition.dot(next_values) for action, transition in self.transitions.items()}
            values = np.zeros(self.n_states)
            for i, state in enumerate(self.states):
                if not self.model.is_terminal(state):
                    actions = self.get_actions(state)
                    values[i] = max(action_value[action][i] for action in actions)
        self.values = values
        # policies
        policy = self.get_policy(self.player)
        adversary_policy = self.get_policy(-self.player)
        return policy, adversary_policy
        

**TicTacToe: Perfect players**

In [ ]:
Game = TicTacToe
game = Game()

algo = ValueIteration(game)
policy, adversary_policy = algo.get_perfect_players()

In [ ]:
game = TicTacToe(adversary_policy=adversary_policy)
agent = Agent(game, policy=policy)

gains = agent.get_gains()
print("Gains of a perfect player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The agent losses against the adversary ~70% of the times, even when switching policies between agent and adversary. One possible explanation it that going first is a disadvantage for the learned policies.

When the `adversary policy` is assigned to both the adversary and the agent the agent has a 100% win rate. When the policy `policy` is assigned to both, the agent has a 0% win rate. This behavior could be explained by the previous explanation. The adversary policy is better because it learns from the plays from a good player (first policy).

**Nim: Perfect players**

In [ ]:
Game = Nim
game = Game()

algo = ValueIteration(game)
policy, adversary_policy = algo.get_perfect_players()

In [ ]:
game = Nim(adversary_policy=adversary_policy)
agent = Agent(game, policy=policy)

gains = agent.get_gains()
print("Gains of a perfect player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

In [ ]:
game = Nim(adversary_policy=policy)
agent = Agent(game, policy=adversary_policy)

gains = agent.get_gains()
print("Gains of a perfect player vs one-step adversary: {}".format(np.mean(gains)))

In [ ]:
results = dict(zip(*np.unique(gains, return_counts=True)))

print("Losses: {}".format(results[-1] if -1 in results else 0))
print("Draws: {}".format(results[0] if 0 in results else 0))
print("Wins: {}".format(results[1] if 1 in results else 0))

The behavior is the same as for TicTacToe, the agent losses ~75% of the times, even when switching policies between agent and adversary.

After doing some research, if two perfect computers play Nim against each other, the outcome depends entirely on the initial configuration of the game. Our initial board configuration of [1, 3, 5, 7] is a losing position for the first player if both players play perfectly. The second player, if perfect, will always be able to force a win. This could be a reason for the high amount of losses. This can be confirmed by switching the policies and observing that the agent still losses ~75% of the time.

**Connect Four: Perfect players**

In [ ]:
from model import ConnectFour

In [ ]:
try:
    game = ConnectFour()
    algo = ValueIteration(game)
except ValueError as error:
    print(error)

This approach is not applicable due to the state space being too large. Value iteration updates the value of every state by computing the expected reward for every possible action and transition. The transition matrix would be too large in space.